# Predicción de causas de falla en una red de distribución eléctrica

**Contexto:** una red de distribución eléctrica registra miles de interrupciones al año. Cada evento trae consigo un dato tabular (circuito, municipio, duración) y un comentario de campo en texto libre escrito por el operador. Este cuaderno construye un clasificador que combina ambos tipos de dato para predecir la causa de la falla.

**Nota sobre los datos:** por confidencialidad, este cuaderno usa un dataset **sintético reducido** (~650 registros) que imita la estructura, tipos de dato y desbalance de clases del proyecto original, pero no contiene ningún dato real de cliente.

Artículo completo con el diseño y las decisiones de este proyecto: *(liga a la plataforma — sección Fallas en redes eléctricas)*

Un enfoque similar, publicado y de acceso abierto: [Enhancing reliability in power grid fault management via multi-task learning-based knowledge graph completion](https://link.springer.com/article/10.1007/s10791-026-10043-x)


## Arquitectura de la solución

- Dos ramas de entrada: **tabular** (categóricas + numérica) y **texto** (comentario de campo).
- Rama tabular: cada categórica pasa por su propio embedding; la numérica se escala y se proyecta a la misma dimensión.
- Rama de texto: DistilBERT tokeniza y codifica el comentario; se toma el embedding del token `[CLS]`.
- Ambas ramas se concatenan y pasan por una capa densa con dropout antes de la capa de salida (3 clases).


In [ ]:
# Diagrama simple de la arquitectura (solo para referencia visual dentro del cuaderno)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 5))
ax.axis('off')

boxes = {
    "Categóricas\n+ Numérica": (0.05, 0.6, 0.25, 0.25),
    "Comentario\n(texto)": (0.05, 0.15, 0.25, 0.25),
    "Embeddings\ntabulares": (0.4, 0.6, 0.25, 0.25),
    "DistilBERT\n[CLS]": (0.4, 0.15, 0.25, 0.25),
    "Concat +\nDense + Dropout": (0.75, 0.4, 0.25, 0.25),
}

for label, (x, y, w, h) in boxes.items():
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02",
                                          linewidth=1.2, edgecolor="#006a87", facecolor="#f4f6f8"))
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=9)

arrows = [((0.3, 0.72), (0.4, 0.72)), ((0.3, 0.27), (0.4, 0.27)),
          ((0.65, 0.72), (0.75, 0.52)), ((0.65, 0.27), (0.75, 0.52))]
for (x1, y1), (x2, y2) in arrows:
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color="#5a6c7d"))

plt.title("Rama tabular + rama de texto → clasificación", fontsize=11)
plt.tight_layout()
plt.show()


## Carga de datos

- Dataset sintético: `fallas_red_electrica_sintetico.csv` (~650 registros, 10 columnas).
- Mismas columnas que el proyecto original: identificadores de placa, elemento, municipio y circuito, fechas de interrupción, duración, causa y comentario de campo.
- Sin valores nulos por diseño (el generador sintético no introduce nulos).


In [ ]:
import pandas as pd

path = "fallas_red_electrica_sintetico.csv"
bd = pd.read_csv(path)

print(bd.shape)
bd.head()


## Explicación de datos

- `CAUSA_FALLA`: 4 categorías originales (Contacto animal, Elemento fallado, Vegetación, Otros).
- `COMENTARIOS_OA`: texto libre del operador, es la señal más rica pero también la más difícil de aprovechar directamente.
- `CKTO_ELE_OPE` / `MUNICIPIO_ELEM_INI`: alta cardinalidad — no se puede tratar como una categórica más sin agrupar antes.
- `DURACION_INT_HRS`: numérica continua, distribución sesgada a la derecha (muchas fallas cortas, pocas muy largas).


In [ ]:
for col in bd.columns:
    print(f"{col:22s} | tipo: {str(bd[col].dtype):10s} | únicos: {bd[col].nunique()}")


## Análisis exploratorio (EDA)

- Revisamos el balance de clases antes de tocar nada — el desbalance real determina si hace falta oversampling.
- Analizamos qué circuitos concentran más interrupciones a lo largo del tiempo: son los que más información aportan al modelo.
- Miramos la distribución de duración de interrupción para decidir cómo escalarla.


In [ ]:
import matplotlib.pyplot as plt

# Balance de clases original
conteo = bd['CAUSA_FALLA'].value_counts()
print(conteo)

conteo.plot(kind='bar', color='#006a87')
plt.title('Distribución original de causas de falla')
plt.ylabel('Registros')
plt.tight_layout()
plt.show()


In [ ]:
# Circuitos con mayor frecuencia sostenida en el tiempo
bd['FECHA_INI_INTERRUP'] = pd.to_datetime(bd['FECHA_INI_INTERRUP'])
bd['PERIODO'] = bd['FECHA_INI_INTERRUP'].dt.to_period('M')

presencia = bd.groupby(['CKTO_ELE_OPE', 'PERIODO']).size().reset_index(name='count')
presencia['presente'] = 1
frecuencia_sostenida = presencia.groupby('CKTO_ELE_OPE')['presente'].sum().sort_values(ascending=False)

print(frecuencia_sostenida.head(10))


In [ ]:
bd['DURACION_INT_HRS'].plot(kind='hist', bins=30, color='#00b76c')
plt.title('Distribución de duración de interrupción (horas)')
plt.xlabel('Horas')
plt.tight_layout()
plt.show()


## Modelado

**Decisión de ingeniería clave:** las 4 causas originales se agrupan en 3 clases con significado de negocio (`Natural`, `Interna/Accidente`, `Otros`) antes de modelar. Menos clases, más señal por clase, patrón más aprendible.

- Se balancean las 3 clases por oversampling.
- Categóricas → embeddings; numérica → escalada y proyectada a la misma dimensión.
- Texto (`COMENTARIOS_OA`) → DistilBERT, se usa el embedding del token `[CLS]`.
- Ambas ramas se concatenan → capa densa con dropout → clasificación.
- Loss ponderado por clase + early stopping, porque el desbalance real nunca desaparece del todo.


In [ ]:
# Agrupar causas originales en 3 clases con significado de negocio
mapa_3clases = {
    "Contacto animal": "Natural",
    "Vegetación": "Natural",
    "Elemento fallado": "Interna/Accidente",
    "Otros": "Otros",
}
bd['CAUSA_FALLA_3CLASSES'] = bd['CAUSA_FALLA'].map(mapa_3clases)
print(bd['CAUSA_FALLA_3CLASSES'].value_counts())


In [ ]:
# Oversampling para balancear las 3 clases
desired_count = bd['CAUSA_FALLA_3CLASSES'].value_counts().max()
df_balanced = pd.DataFrame()

for cls in bd['CAUSA_FALLA_3CLASSES'].unique():
    df_cls = bd[bd['CAUSA_FALLA_3CLASSES'] == cls]
    df_cls_balanced = df_cls.sample(desired_count, replace=True, random_state=42)
    df_balanced = pd.concat([df_balanced, df_cls_balanced])

df_final = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
print(df_final['CAUSA_FALLA_3CLASSES'].value_counts())


In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel

categorical_cols = ['PLACA_INI_INTERRUP', 'PLACA_FIN_INTERRUP', 'ELEM_FIN_INTERRUP',
                    'MUNICIPIO_ELEM_INI', 'CKTO_ELE_OPE']
numerical_cols = ['DURACION_INT_HRS']
text_col = 'COMENTARIOS_OA'
target_col = 'CAUSA_FALLA_3CLASSES'

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_final[col] = le.fit_transform(df_final[col].astype(str))
    label_encoders[col] = le

scaler = StandardScaler()
df_final[numerical_cols] = scaler.fit_transform(df_final[numerical_cols])

target_le = LabelEncoder()
df_final[target_col] = target_le.fit_transform(df_final[target_col])
num_classes = df_final[target_col].nunique()

X_train, X_val = train_test_split(df_final, test_size=0.2, random_state=42, stratify=df_final[target_col])

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class TabTextBERTDataset(Dataset):
    def __init__(self, df, cat_cols, num_cols, text_col, target_col, tokenizer, max_len=64):
        self.cat_data = df[cat_cols].values.astype(np.int64)
        self.num_data = df[num_cols].values.astype(np.float32)
        self.texts = df[text_col].fillna("").values
        self.targets = df[target_col].values.astype(np.int64)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        cat = torch.tensor(self.cat_data[idx])
        num = torch.tensor(self.num_data[idx])
        target = torch.tensor(self.targets[idx])
        encoding = self.tokenizer(self.texts[idx], truncation=True, padding='max_length',
                                   max_length=self.max_len, return_tensors='pt')
        return cat, num, encoding['input_ids'].squeeze(0), encoding['attention_mask'].squeeze(0), target

train_dataset = TabTextBERTDataset(X_train, categorical_cols, numerical_cols, text_col, target_col, tokenizer)
val_dataset = TabTextBERTDataset(X_val, categorical_cols, numerical_cols, text_col, target_col, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)


In [ ]:
class TabBERTClassifier(nn.Module):
    def __init__(self, cat_cardinalities, cat_embed_dim=32, num_features=1,
                 bert_model_name='distilbert-base-uncased', num_classes=3, dropout=0.3):
        super().__init__()
        self.cat_embeddings = nn.ModuleList([nn.Embedding(card, cat_embed_dim) for card in cat_cardinalities])
        self.num_linear = nn.Linear(num_features, cat_embed_dim)
        self.bert = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.fc1 = nn.Linear(self.bert_dim + cat_embed_dim * (len(cat_cardinalities) + 1), 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, cat, num, input_ids, attention_mask):
        cat_embeds = [emb(cat[:, i]) for i, emb in enumerate(self.cat_embeddings)]
        cat_embeds = torch.cat(cat_embeds, dim=1)
        num_embed = self.num_linear(num)
        x_tab = torch.cat([cat_embeds, num_embed], dim=1)
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        bert_emb = bert_out.last_hidden_state[:, 0, :]
        x = self.dropout(self.relu(self.fc1(torch.cat([x_tab, bert_emb], dim=1))))
        return self.fc2(x)

cat_cardinalities = [df_final[col].nunique() for col in categorical_cols]
model = TabBERTClassifier(cat_cardinalities, num_features=len(numerical_cols), num_classes=num_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

class_counts = df_final[target_col].value_counts().sort_index().values
class_weights = (1. / torch.tensor(class_counts, dtype=torch.float32)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)


In [ ]:
epochs = 8
patience = 3
best_val_loss = np.inf
counter = 0
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for cat, num, input_ids, attention_mask, target in train_loader:
        cat, num, input_ids, attention_mask, target = [t.to(device) for t in (cat, num, input_ids, attention_mask, target)]
        optimizer.zero_grad()
        outputs = model(cat, num, input_ids, attention_mask)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for cat, num, input_ids, attention_mask, target in val_loader:
            cat, num, input_ids, attention_mask, target = [t.to(device) for t in (cat, num, input_ids, attention_mask, target)]
            outputs = model(cat, num, input_ids, attention_mask)
            loss_val = criterion(outputs, target)
            val_loss += loss_val.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == target).sum().item()
            total += target.size(0)
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_loss:.4f} - Val Loss: {val_loss:.4f} - Val Acc: {correct/total:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break


## Evaluación

- Cargamos el mejor checkpoint (menor pérdida de validación).
- Revisamos matriz de confusión y reporte de clasificación por clase, no solo el accuracy global — con 3 clases balanceadas, el accuracy solo no dice si el modelo confunde clases específicas.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

model.load_state_dict(torch.load("best_model.pt"))
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for cat, num, input_ids, attention_mask, target in val_loader:
        cat, num, input_ids, attention_mask, target = [t.to(device) for t in (cat, num, input_ids, attention_mask, target)]
        outputs = model(cat, num, input_ids, attention_mask)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(target.cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

acc = (all_preds == all_targets).mean()
print(f"Accuracy de validación: {acc:.4f}\n")

cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_le.classes_, yticklabels=target_le.classes_)
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title('Matriz de confusión')
plt.tight_layout()
plt.show()

print(classification_report(all_targets, all_preds, target_names=[str(c) for c in target_le.classes_]))


## Aplicación rápida del modelo

Antes de pensar en una app de escritorio o un Excel conectado al modelo, conviene tener una función simple dentro del mismo cuaderno que reciba un caso nuevo y devuelva la predicción. Esto ayuda a detectar rápido si el modelo tiene sentido antes de invertir en una interfaz.


In [ ]:
def predecir_falla(circuito, municipio, elemento, placa_ini, placa_fin, duracion_hrs, comentario):
    """Aplica el modelo entrenado a un caso nuevo y regresa la clase predicha con probabilidades."""
    model.eval()

    def encode_safe(le, value):
        return le.transform([value])[0] if value in le.classes_ else 0

    cat_vals = [
        encode_safe(label_encoders['PLACA_INI_INTERRUP'], placa_ini),
        encode_safe(label_encoders['PLACA_FIN_INTERRUP'], placa_fin),
        encode_safe(label_encoders['ELEM_FIN_INTERRUP'], elemento),
        encode_safe(label_encoders['MUNICIPIO_ELEM_INI'], municipio),
        encode_safe(label_encoders['CKTO_ELE_OPE'], circuito),
    ]
    cat_tensor = torch.tensor([cat_vals]).to(device)
    num_tensor = torch.tensor(scaler.transform([[duracion_hrs]]), dtype=torch.float32).to(device)

    encoding = tokenizer(comentario, truncation=True, padding='max_length', max_length=64, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(cat_tensor, num_tensor, input_ids, attention_mask)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]

    clase = target_le.classes_[probs.argmax()]
    return clase, dict(zip(target_le.classes_, probs.round(3)))

# Ejemplo de uso
clase, probs = predecir_falla(
    circuito="Circuito_4", municipio="Municipio_2", elemento="Aislador",
    placa_ini="PLACA_0029", placa_fin="PLACA_0018", duracion_hrs=2.5,
    comentario="Rama sobre la linea provoco interrupcion"
)
print("Clase predicha:", clase)
print("Probabilidades:", probs)


## Hallazgos principales

- Agrupar 4 causas ruidosas en 3 clases con significado de negocio fue más determinante para el desempeño que cualquier ajuste de hiperparámetros.
- El comentario de campo (texto libre) aporta señal que las columnas tabulares por sí solas no capturan, sobre todo para distinguir causas "Interna/Accidente" de "Otros".
- El desbalance original no desaparece solo con oversampling — el loss ponderado por clase sigue siendo necesario en el entrenamiento.
- Tener una función de inferencia simple dentro del propio cuaderno permite validar el modelo con casos nuevos antes de invertir en una interfaz o aplicación final.
